# 18C follow-up — Acquisition boundary audit
Scan only 2019 Q2–Q4 and quantify additional 2018 originations. Preserve the five-file audit. This is not a final coverage or semantic-mapping approval.

## 1. Check archives and reviewed date positions

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, re, time, zipfile, ctypes
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW = ROOT / 'data/raw/Fannie_Mae/release_2026Q2'
EXPECTED_FIELDS = 114
quarters = ['2019Q2', '2019Q3', '2019Q4']
for q in quarters:
    assert (RAW / f'{q}.zip').is_file(), f'Missing {q}.zip'
class MemoryCounters(ctypes.Structure):
    _fields_ = [('cb', ctypes.c_ulong), ('PageFaultCount', ctypes.c_ulong)] + [(name, ctypes.c_size_t) for name in ['PeakWorkingSetSize','WorkingSetSize','QuotaPeakPagedPoolUsage','QuotaPagedPoolUsage','QuotaPeakNonPagedPoolUsage','QuotaNonPagedPoolUsage','PagefileUsage','PeakPagefileUsage']]

def peak_ram_gib():
    try:
        p = MemoryCounters()
        p.cb = ctypes.sizeof(p)
        ctypes.windll.kernel32.GetCurrentProcess.restype = ctypes.c_void_p
        handle = ctypes.windll.kernel32.GetCurrentProcess()
        ok = ctypes.windll.psapi.GetProcessMemoryInfo(ctypes.c_void_p(handle), ctypes.byref(p), p.cb)
        return round(p.PeakWorkingSetSize / 2**30, 4) if ok else None
    except AttributeError:
        return None

# Confirm the origination field position against the reviewed Notebook 15 code.
import ast
source_notebook = ROOT / 'notebooks/15_fannie_external_dataset_construction.ipynb'
assert source_notebook.is_file(), 'Notebook 15 is required to recover its reviewed field positions.'
positions = None
for cell in json.loads(source_notebook.read_text(encoding='utf-8'))['cells']:
    if cell['cell_type'] != 'code':
        continue
    try:
        tree = ast.parse(''.join(cell['source']))
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'POSITIONS' for t in node.targets):
            positions = ast.literal_eval(node.value)
assert isinstance(positions, dict) and 'orig' in positions, 'Review POSITIONS in Notebook 15.'
orig_index = positions['orig'] - 1
assert orig_index == 13, 'Glossary origination position differs from Notebook 15.'
print('Reviewed origination position:', orig_index + 1)
assert 'report' in positions and 'first_payment' in positions
report_index = positions['report'] - 1
payment_index = positions['first_payment'] - 1
assert (report_index, payment_index) == (2, 14), 'Glossary date positions differ from Notebook 15.'
print('Reviewed positions:', {k: positions[k] for k in ['orig','report','first_payment']})

# Preflight every archive before the full scan.
for q in quarters:
    with zipfile.ZipFile(RAW / f'{q}.zip') as z:
        members = [m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.csv')]
        assert len(members) == 1, f'{q}: unexpected archive contents'
        with z.open(members[0]) as stream:
            sample = [stream.readline() for _ in range(100)]
        counts = Counter(len(r.rstrip(b'\r\n').split(b'|')) for r in sample if r)
        print(q, dict(counts))
        assert counts == {EXPECTED_FIELDS: 100}, f'{q}: unexpected layout; review before scanning'


Reviewed origination position: 14
Reviewed positions: {'orig': 14, 'report': 3, 'first_payment': 15}
2019Q2 {114: 100}
2019Q3 {114: 100}
2019Q4 {114: 100}


## 2. Load previous aggregate results

In [2]:
# Load the previous aggregate review; do not scan those five archives again.
from io import BytesIO
import os
import pyarrow.parquet as pq
import pyarrow as pa

PROCESSED = ROOT / 'data/processed/Fannie_Mae'
OLD_OVERRIDE = None  # Set an exact review ZIP path if multiple different versions exist.
previous = [Path(OLD_OVERRIDE)] if OLD_OVERRIDE else sorted((PROCESSED / 'notebook_18c_pilot_schema_audit').rglob('notebook_18c_pilot_schema_review.zip'))
assert previous, 'Locate the successful 18C review ZIP.'
assert len({hashlib.sha256(p.read_bytes()).hexdigest() for p in previous}) == 1, 'Set OLD_OVERRIDE to the reviewed ZIP.'
with zipfile.ZipFile(previous[-1]) as z:
    prior_inventory = pd.read_csv(z.open('archive_inventory.csv'))
    prior_coverage = pd.read_csv(z.open('2018_acquisition_coverage.csv'))
    prior_summary = json.loads(z.read('audit_summary.json'))
assert prior_summary['layout_fields'] == 114
assert prior_summary['cross_file_overlap_occurrences'] == 0
assert set(prior_inventory.acquisition_file) == {'2018Q1','2018Q2','2018Q3','2018Q4','2019Q1'}
for r in prior_inventory.itertuples():
    assert (RAW / f'{r.acquisition_file}.zip').stat().st_size == r.zip_bytes, 'Prior archive size changed; review provenance.'
print('Prior audit loaded:', len(prior_inventory), 'archives. Prior ID overlap cannot be checked without their IDs.')
CHECKPOINT = PROCESSED / 'notebook_18c_boundary_checkpoints_v1'
CHECKPOINT.mkdir(parents=True, exist_ok=True)


Prior audit loaded: 5 archives. Prior ID overlap cannot be checked without their IDs.


## 3. Define the physical scan

In [3]:
date_pattern = re.compile(rb'(0[1-9]|1[0-2])[0-9]{4}')
id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')

def valid_date(value):
    return bool(date_pattern.fullmatch(value)) and 1900 <= int(value[2:]) <= 2100

def scan_archive(q, prior_ids):
    path = RAW / f'{q}.zip'
    signature = (path.stat().st_size, path.stat().st_mtime_ns)
    counts, quality = Counter(), Counter()
    seen = set()
    current = None
    dates, payments = set(), set()
    report_min, report_max = None, None
    sha = hashlib.sha256()
    start = time.perf_counter()
    rows = 0

    def finish(key, dates, payments):
        if key is None:
            return
        if key in seen:
            raise ValueError(f'{q}: noncontiguous loan history; stop for review')
        seen.add(key)
        if len(dates) != 1:
            counts['conflicting_or_missing'] += 1
        else:
            value = next(iter(dates))
            if not valid_date(value):
                counts['invalid_date'] += 1
            else:
                counts[f'{value[2:].decode()}Q{(int(value[:2])-1)//3+1}'] += 1
        if len(payments) != 1 or not valid_date(next(iter(payments), b'')):
            quality['loan_first_payment_unresolved'] += 1

    with zipfile.ZipFile(path) as z:
        members = [m for m in z.infolist() if not m.is_dir() and m.filename.lower().endswith('.csv')]
        assert len(members) == 1, f'{q}: review CSV archive members'
        member = members[0]
        assert Path(member.filename).name.lower() == f'{q.lower()}.csv', f'{q}: unexpected CSV name'
        with z.open(member) as stream:
            for rows, raw in enumerate(stream, 1):
                sha.update(raw)
                parts = raw.rstrip(b'\r\n').split(b'|')
                if len(parts) != EXPECTED_FIELDS:
                    raise ValueError(f'{q}: expected {EXPECTED_FIELDS} fields, found {len(parts)} at row {rows}')
                key = parts[1].strip()
                if not id_pattern.fullmatch(key):
                    raise ValueError(f'{q}: invalid loan ID at row {rows}')
                if key != current:
                    finish(current, dates, payments)
                    current, dates, payments = key, set(), set()
                orig, payment, report = (parts[i].strip() for i in [orig_index, payment_index, report_index])
                if orig: dates.add(orig)
                if payment: payments.add(payment)
                if not valid_date(orig): quality['row_origination_missing_or_invalid'] += 1
                if not valid_date(payment): quality['row_first_payment_missing_or_invalid'] += 1
                if not valid_date(report):
                    quality['row_report_missing_or_invalid'] += 1
                else:
                    period = int(report[2:]) * 100 + int(report[:2])
                    report_min = period if report_min is None else min(period, report_min)
                    report_max = period if report_max is None else max(period, report_max)
                if rows % 5_000_000 == 0:
                    print(f'{q}: {rows:,} rows scanned', flush=True)
            finish(current, dates, payments)
    assert rows > 0 and sum(counts.values()) == len(seen)
    assert signature == (path.stat().st_size, path.stat().st_mtime_ns)
    overlap = len(seen & prior_ids)
    prior_ids.update(seen)
    result = dict(acquisition_file=q, zip_bytes=signature[0], csv_member=member.filename,
                  csv_bytes=member.file_size, physical_rows=rows, unique_loans=len(seen),
                  overlap_with_previous_files=overlap, fields_per_row=EXPECTED_FIELDS,
                  earliest_report=report_min, latest_report=report_max,
                  decompressed_sha256=sha.hexdigest(), scan_seconds=round(time.perf_counter()-start,2),
                  process_peak_working_set_gib=peak_ram_gib())
    print(q, result['unique_loans'], 'loans;', overlap, 'overlaps;', result['scan_seconds'], 'seconds', flush=True)
    return result, counts, quality, seen



## 4. Scan new archives and save checkpoints
Local ID files must stay in ignored data storage. Do not share them.

In [4]:
# Each completed scan is saved independently. Checkpoints include local-only loan IDs.
def archive_signature(path):
    st = path.stat()
    # Hash compressed archive (not the much larger decompressed stream).
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    assert (st.st_size, st.st_mtime_ns) == (path.stat().st_size, path.stat().st_mtime_ns)
    return {'bytes': st.st_size, 'mtime_ns': st.st_mtime_ns, 'sha256': h.hexdigest()}

prior_ids = set()
inventory, distribution, checks = [], [], []
for q in quarters:
    path = RAW / f'{q}.zip'
    sig = archive_signature(path)
    meta = CHECKPOINT / f'{q}.json'
    ids_file = CHECKPOINT / f'{q}_ids.parquet'
    if meta.exists() and ids_file.exists():
        saved = json.loads(meta.read_text(encoding='utf-8'))
        assert saved['source_signature'] == sig, f'{q}: archive changed; preserve old checkpoint and review.'
        assert hashlib.sha256(ids_file.read_bytes()).hexdigest() == saved['ids_sha256'], 'Checkpoint integrity failure.'
        seen = set(pq.read_table(ids_file, columns=['loan_id'])['loan_id'].to_pylist())
        result, counts, quality = saved['result'], saved['counts'], saved['quality']
        assert len(seen) == result['unique_loans'] == sum(counts.values())
        print(q, 'loaded checkpoint', flush=True)
    else:
        result, counts, quality, seen = scan_archive(q, set())
        assert archive_signature(path) == sig, f'{q}: source changed during scan.'
        tmp = ids_file.with_suffix('.tmp')
        pq.write_table(pa.table({'loan_id': sorted(seen)}), tmp, compression='zstd')
        os.replace(tmp, ids_file)
        saved = dict(source_signature=sig, result=result, counts=dict(counts), quality=dict(quality), ids_sha256=hashlib.sha256(ids_file.read_bytes()).hexdigest())
        temp_meta = meta.with_suffix('.tmp')
        temp_meta.write_text(json.dumps(saved, indent=2), encoding='utf-8')
        os.replace(temp_meta, meta)
    result = dict(result)
    result['overlap_with_previous_new_files'] = len(seen & prior_ids)
    result.pop('overlap_with_previous_files', None)
    prior_ids.update(seen)
    inventory.append(result)
    distribution.extend(dict(acquisition_file=q, origination_cohort=k, loans=v) for k,v in sorted(counts.items()))
    checks.extend(dict(acquisition_file=q, check=k, count=quality.get(k,0)) for k in ['loan_first_payment_unresolved','row_origination_missing_or_invalid','row_first_payment_missing_or_invalid','row_report_missing_or_invalid'])
    del seen


2019Q2: 5,000,000 rows scanned
2019Q2: 10,000,000 rows scanned
2019Q2 407389 loans; 0 overlaps; 159.86 seconds
2019Q3: 5,000,000 rows scanned
2019Q3: 10,000,000 rows scanned
2019Q3: 15,000,000 rows scanned
2019Q3: 20,000,000 rows scanned
2019Q3: 25,000,000 rows scanned
2019Q3 673662 loans; 0 overlaps; 299.5 seconds
2019Q4: 5,000,000 rows scanned
2019Q4: 10,000,000 rows scanned
2019Q4: 15,000,000 rows scanned
2019Q4: 20,000,000 rows scanned
2019Q4: 25,000,000 rows scanned
2019Q4 690018 loans; 0 overlaps; 289.11 seconds


## 5. Compare acquisition coverage and export aggregate review
Prior audit did not retain IDs, so overlap against those files remains unverified. Later acquisition files may still contain earlier originations.

In [5]:
inventory_df = pd.DataFrame(inventory)
cohorts = pd.DataFrame(distribution)
quality_df = pd.DataFrame(checks)
cols = ['2018Q1','2018Q2','2018Q3','2018Q4']
coverage = cohorts[cohorts.origination_cohort.isin(cols)].pivot_table(index='acquisition_file', columns='origination_cohort', values='loans', aggfunc='sum', fill_value=0).reindex(index=quarters, columns=cols, fill_value=0)
coverage['2018_total'] = coverage.sum(axis=1)
coverage = coverage.reset_index()
combined = pd.concat([prior_coverage, coverage], ignore_index=True)
assert inventory_df.unique_loans.sum() - inventory_df.overlap_with_previous_new_files.sum() == len(prior_ids)
print('Additional 2018 loan occurrences:', int(coverage['2018_total'].sum()))
print('These counts are not deduplicated against the earlier five files.')
display(inventory_df.drop(columns=['decompressed_sha256']))
display(combined)
display(quality_df)
out = PROCESSED / 'notebook_18c_boundary_audit' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True, exist_ok=False)
for name, table in {'new_archive_inventory.csv':inventory_df,'new_origination_cohort_counts.csv':cohorts,'new_date_quality_checks.csv':quality_df,'combined_2018_acquisition_coverage.csv':combined}.items():
    table.to_csv(out / name, index=False)
summary = dict(prior_review_sha256=hashlib.sha256(previous[-1].read_bytes()).hexdigest(), new_files=quarters, new_distinct_ids=len(prior_ids), overlap_among_new_files=int(inventory_df.overlap_with_previous_new_files.sum()), overlap_with_prior_five_verified=False, additional_2018_loan_occurrences=int(coverage['2018_total'].sum()), prior_2018_loan_occurrences=int(prior_coverage['2018_total'].sum()), coverage_complete=False, semantic_mapping_verified=False, release_consistency_verified=False, sampling_performed=False, targets_constructed=False)
(out/'boundary_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
review = out/'notebook_18c_boundary_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        if f.suffix in {'.csv','.json'}: z.write(f,f.name)
print('Upload only:',review)


Additional 2018 loan occurrences: 5789
These counts are not deduplicated against the earlier five files.


,acquisition_file,zip_bytes,csv_member,csv_bytes,physical_rows,unique_loans,fields_per_row,earliest_report,latest_report,scan_seconds,process_peak_working_set_gib,overlap_with_previous_new_files
0,2019Q2,323678542,2019Q2.csv,4452405737,14649207,407389,114,201904,202606,159.86,0.1699,0
1,2019Q3,653608773,2019Q3.csv,8573683782,26733171,673662,114,201907,202606,299.50,0.2725,0
2,2019Q4,705634501,2019Q4.csv,9305598700,28974968,690018,114,201910,202606,289.11,0.3563,0


,acquisition_file,2018Q1,2018Q2,2018Q3,2018Q4,2018_total
0,2018Q1,260423,0,0,0,260423
1,2018Q2,153395,298262,0,0,451657
2,2018Q3,3494,192765,302953,0,499212
3,2018Q4,180,3361,171034,245754,420329
4,2019Q1,259,322,3629,144102,148312
5,2019Q2,200,343,639,2760,3942
6,2019Q3,80,152,463,678,1373
7,2019Q4,17,39,142,276,474


,acquisition_file,check,count
0,2019Q2,loan_first_payment_unresolved,31
1,2019Q2,row_origination_missing_or_invalid,0
2,2019Q2,row_first_payment_missing_or_invalid,0
3,2019Q2,row_report_missing_or_invalid,0
4,2019Q3,loan_first_payment_unresolved,49
5,2019Q3,row_origination_missing_or_invalid,0
6,2019Q3,row_first_payment_missing_or_invalid,0
7,2019Q3,row_report_missing_or_invalid,0
8,2019Q4,loan_first_payment_unresolved,60
9,2019Q4,row_origination_missing_or_invalid,0


Upload only: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_18c_boundary_audit\20261125T212350398026Z\notebook_18c_boundary_review.zip
